In [ ]:
import os
from google.colab import drive
import pandas as pd
import uuid
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import random
import matplotlib.pyplot as plt

In [ ]:
drive.mount('/content/drive')

In [ ]:

data_directory = "/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset_emotion/"
filenames = os.listdir(data_directory)
post_filenames = sorted([f for f in filenames if "post" in f])
comment_filenames = sorted([f for f in filenames if "comment" in f])
print(filenames)
print(post_filenames)
print(comment_filenames)

In [ ]:
# Load just the first pair of files
posts_df = pd.read_csv(data_directory + post_filenames[0])
comments_df = pd.read_csv(data_directory + comment_filenames[0])

print(f"\nPosts shape: {posts_df.shape}")
print(f"Comments shape: {comments_df.shape}")
print(f"\nPosts columns: {posts_df.columns}")
print(f"Comments columns: {comments_df.columns}")
#print(f"\nPost IDs in posts: {sorted(posts_df['Post_ID'].unique())}")
#print(f"Post IDs in comments: {sorted(comments_df['Post_ID'].unique())}")

In [ ]:
posts_df.head(5)

In [ ]:
import ast
import numpy as np
import pandas as pd


def parse_vec(v):
    """Emotion_Vector may be a stringified list, a list, or an ndarray."""
    if isinstance(v, str):
        v = ast.literal_eval(v)
    return np.asarray(v, dtype=np.float32)


def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())


def _weighted_mean(vecs, words):
    """Word-count-weighted mean over a single unit. Falls back to uniform
    if the unit has zero total words (avoids divide-by-zero)."""
    total = words.sum()
    if total == 0:
        w = np.full(len(vecs), 1.0 / len(vecs), dtype=np.float32)
    else:
        w = words / total
    return (vecs * w[:, None]).sum(axis=0)


def build_samples(posts_df, comments_df, variant="raw", min_len=2):
    """
    Build one sample per post as an ordered (seq_len, 43) float32 array.

    variant:
      "raw"      -> every sentence vector, in order (steps = sentences)
      "mean"     -> plain mean per unit  (steps = post + each comment)
      "weighted" -> word-count-weighted mean per unit (steps = units)

    A "unit" is the post, then each comment in Comment_ID order.
    Samples shorter than min_len are dropped (can't form input/target pair).
    """
    assert variant in {"raw", "mean", "weighted"}
    samples = []

    for pid in sorted(posts_df["Post_ID"].unique()):
        # --- post unit ---
        p = posts_df[posts_df["Post_ID"] == pid].sort_values("Sentence_ID")
        post_vecs = np.stack([parse_vec(v) for v in p["Emotion_Vector"]])
        post_words = np.array([word_count(t) for t in p["Post_Content"]],
                              dtype=np.float32)
        units = [(post_vecs, post_words)]

        # --- comment units, in Comment_ID order ---
        c = comments_df[comments_df["Post_ID"] == pid]
        for cid in sorted(c["Comment_ID"].unique()):
            cc = c[c["Comment_ID"] == cid].sort_values("Sentence_ID")
            cvecs = np.stack([parse_vec(v) for v in cc["Emotion_Vector"]])
            cwords = np.array([word_count(t) for t in cc["Comment_Content"]],
                              dtype=np.float32)
            units.append((cvecs, cwords))

        # --- assemble according to variant ---
        if variant == "raw":
            seq = np.concatenate([vecs for vecs, _ in units], axis=0)
        elif variant == "mean":
            seq = np.stack([vecs.mean(axis=0) for vecs, _ in units])
        else:  # weighted
            seq = np.stack([_weighted_mean(vecs, words) for vecs, words in units])

        seq = seq.astype(np.float32)
        if len(seq) >= min_len:
            samples.append(seq)

    return samples

In [ ]:
class EmotionLSTM(nn.Module):
    """
    Generic next-vector predictor. Consumes (batch, seq_len, 43) and returns
    a prediction for every timestep, so training compares out[:, :-1] against
    input[:, 1:].
    """
    def __init__(self, input_size=43, hidden_size=128, num_layers=2,
                 dropout=0.2, sigmoid_output=True):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, input_size)
        self.sigmoid_output = sigmoid_output

    def forward(self, x, hidden=None):
        """
        x: (batch, seq_len, 43). hidden: optional (h0, c0) for stepwise
        generation later. Returns (predictions, hidden).
        """
        out, hidden = self.lstm(x, hidden)          # (batch, seq_len, hidden)
        out = self.head(out)                        # (batch, seq_len, 43)
        if self.sigmoid_output:
            out = torch.sigmoid(out)                # keep in [0, 1]
        return out, hidden

In [ ]:
print(f"\nPosts shape: {posts_df.shape}")
print(f"Comments shape: {comments_df.shape}")

In [ ]:
# ---- 1. Build samples (file 0) ------------------------------------------
VARIANT = "raw"   # "raw" | "mean" | "weighted"

samples = build_samples(posts_df, comments_df, variant=VARIANT, min_len=2)

lengths = [len(s) for s in samples]
print(f"variant = {VARIANT}")
print(f"  samples:         {len(samples)}")
print(f"  vector dim:      {samples[0].shape[1]}")
print(f"  seq_len min/avg/max: {min(lengths)} / {np.mean(lengths):.1f} / {max(lengths)}")
print(f"  total timesteps: {sum(lengths)}")

In [ ]:
PROJECT_SAMPLES = False

# GoEmotions sentiment grouping (dims 0-27) + custom (28-42).
# 'amb' = ambiguous-valence in the GoEmotions scheme. You asked for 3 classes,
# so amb is folded into neutral below — but it's a real 4th category if you want it.
VALENCE = {
    # --- GoEmotions 0-27 (authors' official sentiment mapping) ---
    0:'pos', 1:'pos', 2:'neg', 3:'neg', 4:'pos', 5:'pos',
    6:'amb', 7:'amb', 8:'pos', 9:'neg', 10:'neg', 11:'neg',
    12:'neg', 13:'pos', 14:'neg', 15:'pos', 16:'neg', 17:'pos',
    18:'pos', 19:'neg', 20:'pos', 21:'pos', 22:'amb', 23:'pos',
    24:'neg', 25:'neg', 26:'amb', 27:'neu',
    # --- custom 28-42 (my assignments — review the flagged ones) ---
    28:'neg',   # worry
    29:'pos',   # happiness
    30:'pos',   # fun
    31:'neg',   # hate
    32:'pos',   # autonomy      ⚠ a need, not a valence
    33:'pos',   # safety        ⚠ need
    34:'pos',   # understanding ⚠ need / arguably neutral
    35:'neg',   # empty
    36:'pos',   # enthusiasm
    37:'pos',   # recreation    ⚠ need / arguably neutral
    38:'pos',   # sense of belonging ⚠ need
    39:'pos',   # meaning       ⚠ need
    40:'pos',   # sustenance    ⚠ need — hard to valence at all
    41:'pos',   # creativity    ⚠ need
    42:'neg',   # boredom
}
FOLD_AMB_INTO = 'neu'   # set to keep amb separate if you go 4-way

# Project the 43 dimension into 3 dimensions
def build_projection(valence, fold_amb_into='neu'):
    classes = ['neg', 'neu', 'pos']
    col = {c: i for i, c in enumerate(classes)}
    P = np.zeros((43, 3), dtype=np.float32)
    counts = {c: 0 for c in classes}
    for dim, grp in valence.items():
        if grp == 'amb':
            grp = fold_amb_into
        counts[grp] += 1
    for dim, grp in valence.items():
        if grp == 'amb':
            grp = fold_amb_into
        P[dim, col[grp]] = 1.0 / counts[grp]     # average within group
    return P   # (43, 3)

P = build_projection(VALENCE, FOLD_AMB_INTO)

def project_samples(samples, P):
    return [ (s @ P).astype(np.float32) for s in samples ]   # (seq_len, 3)

if (PROJECT_SAMPLES):
    samples_projected = project_samples(samples, P)
    DIM_SIZE = 3
else:
    samples_projected = samples.copy()
    DIM_SIZE = 43
print(f"3d samples: {len(samples_projected)}  dim: {samples_projected[0].shape[1]}")
print(f"example row (neg,neu,pos): {samples_projected[0][0]}")

In [ ]:
# ---- 2. Post-level 80/20 split (whole samples held out, no leakage) ------
random.seed(42)
idx = list(range(len(samples_projected)))
random.shuffle(idx)
split = int(0.8 * len(idx))
train_samples = [samples_projected[i] for i in idx[:split]]
test_samples  = [samples_projected[i] for i in idx[split:]]
print(f"train samples: {len(train_samples)}   test samples: {len(test_samples)}")

In [ ]:
# ---- 3. Dataset + padded collate ----------------------------------------
class EmotionDataset(Dataset):
    def __init__(self, samples):
        self.inputs  = [torch.from_numpy(s[:-1]) for s in samples]
        self.targets = [torch.from_numpy(s[1:])  for s in samples]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


def collate(batch):
    inputs, targets = zip(*batch)
    lengths = torch.tensor([len(x) for x in inputs])
    inputs  = pad_sequence(inputs,  batch_first=True)   # (B, T, 43)
    targets = pad_sequence(targets, batch_first=True)   # (B, T, 43)
    maxT = inputs.size(1)
    mask = torch.arange(maxT)[None, :] < lengths[:, None]   # (B, T) True = real
    return inputs, targets, mask, lengths


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(EmotionDataset(train_samples), batch_size=32,
                          shuffle=True, collate_fn=collate)
test_loader  = DataLoader(EmotionDataset(test_samples), batch_size=32,
                          shuffle=False, collate_fn=collate)

In [ ]:
# ---- 4. Metrics ----------------------------------------------------------
def step_metrics(preds, targets):
    """Per-timestep loss and accuracy, each shape (B, T)."""
    step_loss = ((preds - targets) ** 2).mean(dim=-1)          # MSE over 43 dims
    step_acc  = 1.0 - (preds - targets).abs().mean(dim=-1)     # 1 - MAE  -> [0,1]
    # alternatives to swap in for step_acc:
    #   cosine:    F.cosine_similarity(preds, targets, dim=-1)
    #   tolerance: ((preds - targets).abs() < 0.1).float().mean(dim=-1)
    return step_loss, step_acc


@torch.no_grad()
def evaluate(model, loader):
    """Returns (avg_loss, avg_acc, per_sample_acc) over real steps only.
    per_sample_acc is a list of 1D arrays: the per-step accuracy of each sample."""
    model.eval()
    tot_loss = tot_acc = tot_steps = 0.0
    per_sample_acc = []
    for inputs, targets, mask, lengths in loader:
        inputs, targets, mask = inputs.to(device), targets.to(device), mask.to(device)
        preds, _ = model(inputs)
        step_loss, step_acc = step_metrics(preds, targets)
        m = mask.float()
        tot_loss  += (step_loss * m).sum().item()
        tot_acc   += (step_acc  * m).sum().item()
        tot_steps += m.sum().item()
        sa = step_acc.cpu().numpy()
        for i, L in enumerate(lengths.tolist()):
            per_sample_acc.append(sa[i, :L])
    return tot_loss / tot_steps, tot_acc / tot_steps, per_sample_acc


def positional_curve(per_sample_acc, grid_size=20):
    """Average per-step accuracy onto a common normalized axis (0=first
    prediction, 1=last), interpolating each variable-length sample."""
    grid = np.linspace(0, 1, grid_size)
    curves = []
    for acc in per_sample_acc:
        if len(acc) == 1:
            curves.append(np.full(grid_size, acc[0]))
        else:
            curves.append(np.interp(grid, np.linspace(0, 1, len(acc)), acc))
    return grid, np.mean(curves, axis=0)

In [ ]:
# ---- 5. Train, evaluating both sets each epoch ---------------------------
model = EmotionLSTM(input_size=DIM_SIZE, hidden_size=128, num_layers=2,
                    dropout=0.2, sigmoid_output=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS, GRAD_CLIP, GRID = 20, 1.0, 20
hist = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
pos_curves = []   # one averaged test curve per epoch

for epoch in range(1, EPOCHS + 1):
    model.train()
    for inputs, targets, mask, lengths in train_loader:
        inputs, targets, mask = inputs.to(device), targets.to(device), mask.to(device)
        preds, _ = model(inputs)
        step_loss, _ = step_metrics(preds, targets)
        loss = (step_loss * mask).sum() / mask.sum()
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

    tr_loss, tr_acc, _              = evaluate(model, train_loader)
    te_loss, te_acc, te_persample  = evaluate(model, test_loader)
    hist["train_loss"].append(tr_loss); hist["test_loss"].append(te_loss)
    hist["train_acc"].append(tr_acc);   hist["test_acc"].append(te_acc)
    pos_grid, curve = positional_curve(te_persample, GRID)
    pos_curves.append(curve)
    print(f"epoch {epoch:3d}  train loss {tr_loss:.5f} acc {tr_acc:.4f}  "
          f"test loss {te_loss:.5f} acc {te_acc:.4f}")

In [ ]:
# ---- 6. Graph 1: loss & accuracy vs epoch --------------------------------
epochs = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(epochs, hist["train_loss"], label="train")
ax1.plot(epochs, hist["test_loss"],  label="test")
ax1.set_xlabel("epoch"); ax1.set_ylabel("MSE loss"); ax1.set_title("Loss"); ax1.legend()

ax2.plot(epochs, hist["train_acc"], label="train")
ax2.plot(epochs, hist["test_acc"],  label="test")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy (1 - MAE)")
ax2.set_title("Accuracy"); ax2.legend()
fig.tight_layout()
plt.show()


# ---- 6. Graph 2: accuracy over the course of a post, per epoch -----------
# 30 lines is unreadable, so plot ~8 epochs spread across training.
n_lines = min(8, EPOCHS)
plot_epochs = np.unique(np.linspace(0, EPOCHS - 1, n_lines).astype(int))
cmap = plt.get_cmap("viridis")

plt.figure(figsize=(8, 5))
for j, e in enumerate(plot_epochs):
    plt.plot(pos_grid, pos_curves[e],
             color=cmap(j / max(1, len(plot_epochs) - 1)),
             label=f"epoch {e + 1}")
plt.xlabel("position in post  (0 = first prediction, 1 = last)")
plt.ylabel("accuracy (1 - MAE)")
plt.title("Test accuracy over the course of a post, by epoch")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def baseline_eval(loader):
    sums = {"Mean": [0.0, 0.0], "Persistence": [0.0, 0.0]}
    steps = 0.0
    for inputs, targets, mask, lengths in loader:
        inputs, targets, mask = inputs.to(device), targets.to(device), mask.to(device)
        m = mask.float()

        # (a) always predict the global training mean
        pred_mean = mean_vec.view(1, 1, -1).expand_as(targets)
        l, a = step_metrics(pred_mean, targets)
        sums["Mean"][0] += (l * m).sum().item()
        sums["Mean"][1] += (a * m).sum().item()

        # (b) persistence: predict the previous vector (== input)
        l, a = step_metrics(inputs, targets)
        sums["Persistence"][0] += (l * m).sum().item()
        sums["Persistence"][1] += (a * m).sum().item()

        steps += m.sum().item()

    for k, (lo, ac) in sums.items():
        print(f"  {k:12s}  loss {lo/steps:.5f}  acc {ac/steps:.4f}")

In [ ]:
# rebuild split from samples_3d, then:
all_train_3d = np.concatenate(train_samples, axis=0)
mean_vec = torch.tensor(all_train_3d.mean(axis=0), dtype=torch.float32, device=device)
print("Baselines on 3D target:")
baseline_eval(test_loader)          # your existing baseline fn
# and the MLP probe with d=3

In [ ]:
import torch.nn.functional as F

# --- build (prev, next) pairs, no sequence structure ---------------------
def make_pairs(samples):
    X = np.concatenate([s[:-1] for s in samples], axis=0)   # (N, 43)
    Y = np.concatenate([s[1:]  for s in samples], axis=0)   # (N, 43)
    return torch.from_numpy(X), torch.from_numpy(Y)

Xtr, Ytr = make_pairs(train_samples)
Xte, Yte = make_pairs(test_samples)
Xtr, Ytr, Xte, Yte = [t.to(device) for t in (Xtr, Ytr, Xte, Yte)]
print(f"pairs: train {len(Xtr)}  test {len(Xte)}")

class MLP(nn.Module):
    def __init__(self, d=43, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, d),
        )
    def forward(self, x):
        return self.net(x)

mlp = MLP(d=43).to(device)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)

for epoch in range(1, 41):
    mlp.train()
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), 256):
        b = perm[i:i+256]
        pred = mlp(Xtr[b])
        loss = F.mse_loss(pred, Ytr[b])
        opt.zero_grad(); loss.backward(); opt.step()

    if epoch % 5 == 0:
        mlp.eval()
        with torch.no_grad():
            pr = mlp(Xte)
            l = F.mse_loss(pr, Yte).item()
            a = (1.0 - (pr - Yte).abs().mean(dim=-1)).mean().item()
        print(f"epoch {epoch:2d}  test loss {l:.5f}  acc {a:.4f}")